In [102]:
# Random Forest for Insurance Type Prediction

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

nhanes_clean_adult = pd.read_pickle("nhanes_clean_adult.pkl")
feature_cols = [
    'age',
    'gender',
    'race',
    'education',
    'income_poverty_ratio'
]
target_col = 'insurance_type'

X = nhanes_clean_adult[feature_cols].copy()
y = nhanes_clean_adult[target_col].copy()

data = pd.concat([X, y], axis=1).dropna(subset=feature_cols + [target_col])
X = data[feature_cols]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features = ['gender', 'race', 'education']
numeric_features = ['age', 'income_poverty_ratio']

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# Define Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', rf)   
])

#StratifiedKFold Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# K=5 Cross Validation Result for Preliminary Analysis
base_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy'
)

print("Base Random Forest - 5-fold CV accuracy:", base_scores)
print("Base mean CV accuracy:", base_scores.mean(), "\n")

#GridSearchCV for Estimator Optimization
param_grid = {
    'clf__n_estimators': [200, 300, 400],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5],
    'clf__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,                    # StratifiedKFold
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best CV accuracy from grid search:", grid.best_score_)
print("Best parameters:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")

#Best Pipeline
best_rf_pipeline = grid.best_estimator_

# ============================================
# 8. （可选）以后最终选模型后，再在 test 上评估
# ============================================
# best_rf_pipeline.fit(X_train, y_train)
# y_pred = best_rf_pipeline.predict(X_test)
# print("\nTest accuracy:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))

Base Random Forest - 5-fold CV accuracy: [0.57931034 0.58007663 0.58358896 0.57515337 0.57208589]
Base mean CV accuracy: 0.5780430388077945 

Best CV accuracy from grid search: 0.5910780857015255
Best parameters:
  clf__max_depth: 20
  clf__max_features: sqrt
  clf__min_samples_split: 5
  clf__n_estimators: 200


In [103]:
# To improve the model accurary, considering adding more soci-economic features
# marital_status (DMDMARTZ) from DEMO_L.xpt
# household_size (DMDHHSIZ) from DEMO_L.xpt
# is_employed (OCQ180) from OCQ_L.xpt

import pandas as pd
import numpy as np

path = "/Users/sherrywang/Desktop/Data_Mining/Project/data-mining-project-starter/data/raw/"
demo = pd.read_sas(path + "DEMO_L.xpt")
ocq  = pd.read_sas(path + "OCQ_L.xpt")

demo_sub = demo[['SEQN', 'DMDMARTZ', 'DMDHHSIZ']].copy()

demo_sub = demo_sub.rename(columns={
    'DMDMARTZ': 'marital_status',
    'DMDHHSIZ': 'household_size'
})

# Marital_status Mapping
marital_map = {
    1: "Married / living with partner",
    2: "Widowed / divorced / separated",
    3: "Never married",
    77: None,
    99: None
}
demo_sub['marital_status'] = demo_sub['marital_status'].map(marital_map)

#Is_employed Mapping
ocq_sub = ocq[['SEQN', 'OCQ180']].copy()
ocq_sub = ocq_sub.rename(columns={'OCQ180': 'hours_worked_last_week'})
ocq_sub['hours_worked_last_week'] = ocq_sub['hours_worked_last_week'].replace(
    {77777: np.nan, 99999: np.nan}
)

# Simple employment indicator
ocq_sub['is_employed'] = ocq_sub['hours_worked_last_week'].apply(
    lambda x: "Yes" if pd.notna(x) and x > 0 else "No"
)

# Merge as the soci_raw

soci_raw = (
    demo_sub
    .merge(ocq_sub, on='SEQN', how='left')
)
soci_raw = soci_raw.rename(columns={'SEQN': 'id'})

soci_raw.head()

,id,marital_status,household_size,hours_worked_last_week,is_employed
0,130378.0,Married / living with partner,4.0,40.0,Yes
1,130379.0,Married / living with partner,2.0,32.0,Yes
2,130380.0,Married / living with partner,7.0,8.0,Yes
3,130381.0,NaN,2.0,NaN,NaN
4,130382.0,NaN,4.0,NaN,NaN


In [104]:
nhanes_clean_adult = pd.read_csv("nhanes_clean_adult.csv")

nhanes_clean_adult_add = nhanes_clean_adult.copy()
nhanes_clean_adult_add = nhanes_clean_adult_add.merge(soci_raw, on='id', how='left')
print('Missing Percentage by Variable')
print(nhanes_clean_adult_add[['marital_status', 'household_size', 'is_employed']].isna().mean().sort_values(ascending=False))

print('\nVariable Types (dtypes)')
print(nhanes_clean_adult_add[['marital_status', 'household_size', 'is_employed']].dtypes)

nhanes_clean_adult_add[['marital_status', 'household_size', 'is_employed']].describe(include='all').round(2)

Missing Percentage by Variable
marital_status    0.045382
household_size    0.000000
is_employed       0.000000
dtype: float64

Variable Types (dtypes)
marital_status     object
household_size    float64
is_employed        object
dtype: object


,marital_status,household_size,is_employed
count,7783,8153.00,8153
unique,3,NaN,2
top,Married / living with partner,NaN,Yes
freq,4136,NaN,4118
mean,NaN,2.62,NaN
std,NaN,1.49,NaN
min,NaN,1.00,NaN
25%,NaN,2.00,NaN
50%,NaN,2.00,NaN
75%,NaN,4.00,NaN


In [105]:
# Handling missing values
nhanes_clean_adult_add['marital_status'] = (
    nhanes_clean_adult_add['marital_status']
        .astype('string')
        .str.strip()
        .fillna("Unknown")
)
print('Missing Percentage by Variable')
print(nhanes_clean_adult_add[['marital_status', 'household_size', 'is_employed']].isna().mean().sort_values(ascending=False))
nhanes_clean_adult_add = nhanes_clean_adult_add.drop(columns=['hours_worked_last_week'], errors='ignore')
nhanes_clean_adult_add = nhanes_clean_adult_add.loc[:, ~nhanes_clean_adult_add.columns.str.contains('^Unnamed')]
nhanes_clean_adult_add.to_csv("nhanes_clean_adult_add.csv", index=False)

Missing Percentage by Variable
marital_status    0.0
household_size    0.0
is_employed       0.0
dtype: float64


In [106]:
# Random Forest for Insurance Type Prediction with more soci-econimic features

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

feature_cols = [
    'age',
    'gender',
    'race',
    'education',
    'income_poverty_ratio',
    'marital_status',
    'household_size',
    'is_employed'
]
target_col = 'insurance_type'

X = nhanes_clean_adult_add[feature_cols].copy()
y = nhanes_clean_adult_add[target_col].copy()

data = pd.concat([X, y], axis=1).dropna(subset=feature_cols + [target_col])
X = data[feature_cols]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features = ['gender', 'race', 'education', 'marital_status', 'is_employed']
numeric_features = ['age', 'income_poverty_ratio','household_size']

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# Define Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', rf)   
])

#StratifiedKFold Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# K=5 Cross Validation Result for Preliminary Analysis
base_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy'
)

print("Base Random Forest - 5-fold CV accuracy:", base_scores)
print("Base mean CV accuracy:", base_scores.mean(), "\n")

#GridSearchCV for Estimator Optimization

param_grid = {
    'clf__n_estimators': [200, 300, 400],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5],
    'clf__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,                    # StratifiedKFold
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best CV accuracy from grid search:", grid.best_score_)
print("Best parameters:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")

#Best Pipeline
best_rf_pipeline = grid.best_estimator_
# ============================================
# 8. （可选）以后最终选模型后，再在 test 上评估
# ============================================
# best_rf_pipeline.fit(X_train, y_train)
# y_pred = best_rf_pipeline.predict(X_test)
# print("\nTest accuracy:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))

Base Random Forest - 5-fold CV accuracy: [0.62452107 0.59693487 0.62653374 0.60812883 0.59509202]
Base mean CV accuracy: 0.6102421079848624 

Best CV accuracy from grid search: 0.6119281667959476
Best parameters:
  clf__max_depth: 20
  clf__max_features: sqrt
  clf__min_samples_split: 2
  clf__n_estimators: 400


In [107]:
# Maximize Random Forest for Insurance Type Prediction with more features

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

feature_cols = [
    'age',
    'gender',
    'race',
    'education',
    'income_poverty_ratio',
    'marital_status',
    'household_size',
    'is_employed',
    # new features
    'smoking_current',
    'diabetes_self_report',
    'bmi',
    'sbp',
    'dbp',
    'a1c',
    'sleep_hours'
]
target_col = 'insurance_type'

X = nhanes_clean_adult_add[feature_cols].copy()
y = nhanes_clean_adult_add[target_col].copy()

data = pd.concat([X, y], axis=1).dropna(subset=feature_cols + [target_col])
X = data[feature_cols]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features = [
    'gender',
    'race',
    'education',
    'marital_status',
    'is_employed',
    'smoking_current',
    'diabetes_self_report'
]
numeric_features = [
    'age',
    'income_poverty_ratio',
    'household_size',
    'bmi',
    'sbp',
    'dbp',
    'a1c',
    'sleep_hours'
]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# Define Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', rf)   
])

#StratifiedKFold Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# K=5 Cross Validation Result for Preliminary Analysis
base_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy'
)

print("Base Random Forest - 5-fold CV accuracy:", base_scores)
print("Base mean CV accuracy:", base_scores.mean(), "\n")

#GridSearchCV for Estimator Optimization

param_grid = {
    'clf__n_estimators': [200, 300, 400],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5],
    'clf__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,                    # StratifiedKFold
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best CV accuracy from grid search:", grid.best_score_)
print("Best parameters:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")

#Best Pipeline
best_rf_pipeline = grid.best_estimator_
# ============================================
# 8. （可选）以后最终选模型后，再在 test 上评估
# ============================================
# best_rf_pipeline.fit(X_train, y_train)
# y_pred = best_rf_pipeline.predict(X_test)
# print("\nTest accuracy:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))

Base Random Forest - 5-fold CV accuracy: [0.64750958 0.64214559 0.65030675 0.65644172 0.6196319 ]
Base mean CV accuracy: 0.6432071081023906 

Best CV accuracy from grid search: 0.6488811320311215
Best parameters:
  clf__max_depth: None
  clf__max_features: sqrt
  clf__min_samples_split: 5
  clf__n_estimators: 400


In [108]:
import pandas as pd

pre = best_rf_pipeline.named_steps['preprocess']
rf  = best_rf_pipeline.named_steps['clf']

feat_names = pre.get_feature_names_out()
importances = rf.feature_importances_

feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)
print(feat_imp.head(20))

num__age                                             0.174678
num__income_poverty_ratio                            0.125205
num__sbp                                             0.071555
num__bmi                                             0.071487
num__dbp                                             0.065294
num__sleep_hours                                     0.060167
num__a1c                                             0.058003
num__household_size                                  0.045709
cat__race_Non-Hispanic White                         0.025679
cat__is_employed_Yes                                 0.022976
cat__marital_status_Unknown                          0.021275
cat__is_employed_No                                  0.020765
cat__education_College graduate or above             0.018896
cat__education_Unknown                               0.017114
cat__marital_status_Married / living with partner    0.014030
cat__smoking_current_Yes                             0.013006
cat__edu

In [109]:
# Maximize Random Forest for Insurance Type

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

feature_cols = [
    'age',
    'gender',
    'race',
    'education',
    'income_poverty_ratio',
    'marital_status',
    'household_size',
    'is_employed',
    # new features
    'smoking_current',
    'diabetes_self_report',
    'bmi',
    'sbp',
    'dbp',
    'a1c',
    'sleep_hours'
]
target_col = 'insurance_type'

X = nhanes_clean_adult_add[feature_cols].copy()
y = nhanes_clean_adult_add[target_col].copy()

data = pd.concat([X, y], axis=1).dropna(subset=feature_cols + [target_col])
X = data[feature_cols]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features = [
    'gender',
    'race',
    'education',
    'marital_status',
    'is_employed',
    'smoking_current',
    'diabetes_self_report'
]
numeric_features = [
    'age',
    'income_poverty_ratio',
    'household_size',
    'bmi',
    'sbp',
    'dbp',
    'a1c',
    'sleep_hours'
]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# Define Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', rf)   
])

#StratifiedKFold Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# K=5 Cross Validation Result for Preliminary Analysis
base_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy'
)

print("Base Random Forest - 5-fold CV accuracy:", base_scores)
print("Base mean CV accuracy:", base_scores.mean(), "\n")

#GridSearchCV for Estimator Optimization

param_grid = {
    'clf__n_estimators': [200, 300, 400],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5],
    'clf__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,                    # StratifiedKFold
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best CV accuracy from grid search:", grid.best_score_)
print("Best parameters:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")

#Best Pipeline
best_rf_pipeline = grid.best_estimator_
# ============================================
# 8. （可选）以后最终选模型后，再在 test 上评估
# ============================================
# best_rf_pipeline.fit(X_train, y_train)
# y_pred = best_rf_pipeline.predict(X_test)
# print("\nTest accuracy:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))

Base Random Forest - 5-fold CV accuracy: [0.64750958 0.64214559 0.65030675 0.65644172 0.6196319 ]
Base mean CV accuracy: 0.6432071081023906 

Best CV accuracy from grid search: 0.6488811320311215
Best parameters:
  clf__max_depth: None
  clf__max_features: sqrt
  clf__min_samples_split: 5
  clf__n_estimators: 400


In [110]:
# Random Forest for Hypertension ： binary variable
def classify_hypertension_binary(row):
    sbp = row['sbp']
    dbp = row['dbp']

    if pd.isna(sbp) or pd.isna(dbp):
        return None

    # Hypertension if SBP ≥ 130 or DBP ≥ 80
    if sbp >= 130 or dbp >= 80:
        return 1
    else:
        return 0

nhanes_clean_adult_add['hypertension'] = nhanes_clean_adult_add.apply(
    classify_hypertension_binary, axis=1
)

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

feature_cols = [
    'age',
    'gender',
    'race',
    'education',
    'income_poverty_ratio',
    'marital_status',
    'household_size',
    'is_employed',
    'smoking_current',
    'diabetes_self_report',
    'bmi',
    'a1c',
    'sleep_hours',
    'insurance_type'
]
target_col = 'hypertension'

X = nhanes_clean_adult_add[feature_cols].copy()
y = nhanes_clean_adult_add[target_col].copy()

data = pd.concat([X, y], axis=1).dropna(subset=feature_cols + [target_col])
X = data[feature_cols]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features = [
    'gender',
    'race',
    'education',
    'marital_status',
    'is_employed',
    'smoking_current',
    'diabetes_self_report',
    'insurance_type'
]
numeric_features = [
    'age',
    'income_poverty_ratio',
    'household_size',
    'bmi',
    'a1c',
    'sleep_hours'
]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# Define Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', rf)   
])

#StratifiedKFold Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# K=5 Cross Validation Result for Preliminary Analysis
base_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy'
)

print("Base Random Forest - 5-fold CV accuracy:", base_scores)
print("Base mean CV accuracy:", base_scores.mean(), "\n")

#GridSearchCV for Estimator Optimization

param_grid = {
    'clf__n_estimators': [200, 300, 400],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5],
    'clf__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,                    # StratifiedKFold
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best CV accuracy from grid search:", grid.best_score_)
print("Best parameters:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")

#Best Pipeline
best_rf_pipeline = grid.best_estimator_
# ============================================
# 8. （可选）以后最终选模型后，再在 test 上评估
# ============================================
# best_rf_pipeline.fit(X_train, y_train)
# y_pred = best_rf_pipeline.predict(X_test)
# print("\nTest accuracy:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))

Base Random Forest - 5-fold CV accuracy: [0.69655172 0.66896552 0.68251534 0.68711656 0.71319018]
Base mean CV accuracy: 0.6896678654537762 

Best CV accuracy from grid search: 0.6950327903532896
Best parameters:
  clf__max_depth: 20
  clf__max_features: sqrt
  clf__min_samples_split: 2
  clf__n_estimators: 200


In [112]:
# Binary Version
from sklearn.model_selection import cross_validate

scoring = {
    'acc': 'accuracy',
    'recall_macro': 'recall_macro',
    'recall_weighted': 'recall_weighted'
}

cv_results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("CV accuracy:", cv_results['test_acc'])
print("CV macro recall:", cv_results['test_recall_macro'])
print("CV weighted recall:", cv_results['test_recall_weighted'])

print("\nMean accuracy:", cv_results['test_acc'].mean())
print("Mean macro recall:", cv_results['test_recall_macro'].mean())
print("Mean weighted recall:", cv_results['test_recall_weighted'].mean())

CV accuracy: [0.65057471 0.6651341  0.66871166 0.66871166 0.67254601]
CV macro recall: [0.36747261 0.39524594 0.380243   0.39372381 0.39694151]
CV weighted recall: [0.65057471 0.6651341  0.66871166 0.66871166 0.67254601]

Mean accuracy: 0.6651356274827822
Mean macro recall: 0.3867253754130049
Mean weighted recall: 0.6651356274827822


In [116]:
#Random Forest for Hypertension ： Normal(+High-normal), Stage 1, Stage 2
# Optimization: Oversampling + Class Weighted
def classify_hypertension_stage(row):
    sbp = row['sbp']
    dbp = row['dbp']

    if pd.isna(sbp) or pd.isna(dbp):
        return None  

    # Stage 2
    if sbp >= 140 or dbp >= 90:
        return "Stage 2"
    # Stage 1
    elif 130 <= sbp < 140 or 80 <= dbp < 90:
        return "Stage 1"
    # Normal + High-normal
    else:
        return "Normal"

nhanes_clean_adult_add['hypertension_stage'] = nhanes_clean_adult_add.apply(
    classify_hypertension_stage, axis=1
)


import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

# imblearn for oversampling
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline  # 用 imblearn 的 Pipeline，把 SMOTE 串进来

feature_cols = [
    'age',
    'gender',
    'race',
    'education',
    'income_poverty_ratio',
    'marital_status',
    'household_size',
    'is_employed',
    'smoking_current',
    'diabetes_self_report',
    'bmi',
    'a1c',
    'sleep_hours',
    'insurance_type'
]

target_col = 'hypertension_stage'

X = nhanes_clean_adult_add[feature_cols].copy()
y = nhanes_clean_adult_add[target_col].copy()

data = pd.concat([X, y], axis=1).dropna(subset=feature_cols + [target_col])
X = data[feature_cols]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features = [
    'gender',
    'race',
    'education',
    'marital_status',
    'is_employed',
    'smoking_current',
    'diabetes_self_report',
    'insurance_type'
]
numeric_features = [
    'age',
    'income_poverty_ratio',
    'household_size',
    'bmi',
    'a1c',
    'sleep_hours'
]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# Define Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',  # class_weight open
    n_jobs=-1
)

# preprocess → SMOTE → RF
rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('smote', SMOTE(random_state=42)),  # oversampling
    ('clf', rf)
])

# StratifiedKFold Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# K=5 Cross Validation Result for Preliminary Analysis
base_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy'
)

print("Base Random Forest - 5-fold CV accuracy:", base_scores)
print("Base mean CV accuracy:", base_scores.mean(), "\n")

# GridSearchCV for Estimator Optimization
param_grid = {
    'clf__n_estimators': [200, 300, 400],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5],
    'clf__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,                    # StratifiedKFold
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best CV accuracy from grid search:", grid.best_score_)
print("Best parameters:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")

# Best Pipeline
best_rf_pipeline = grid.best_estimator_

# ============================================
# 以后如果要在 test 上做最终评估：
# ============================================
# from sklearn.metrics import accuracy_score, classification_report
# best_rf_pipeline.fit(X_train, y_train)
# y_pred = best_rf_pipeline.predict(X_test)
# print("\nTest accuracy:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))


Base Random Forest - 5-fold CV accuracy: [0.64904215 0.6651341  0.66947853 0.66641104 0.67101227]
Base mean CV accuracy: 0.6642156171403051 

Best CV accuracy from grid search: 0.6663617986507769
Best parameters:
  clf__max_depth: None
  clf__max_features: sqrt
  clf__min_samples_split: 2
  clf__n_estimators: 200


In [125]:
# 3 Classes Version
from sklearn.model_selection import cross_validate

scoring = {
    'acc': 'accuracy',
    'recall_macro': 'recall_macro',
    'recall_weighted': 'recall_weighted'
}

cv_results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("CV accuracy:", cv_results['test_acc'])
print("CV macro recall:", cv_results['test_recall_macro'])
print("CV weighted recall:", cv_results['test_recall_weighted'])

print("\nMean accuracy:", cv_results['test_acc'].mean())
print("Mean macro recall:", cv_results['test_recall_macro'].mean())
print("Mean weighted recall:", cv_results['test_recall_weighted'].mean())

CV accuracy: [0.65057471 0.6651341  0.66871166 0.66871166 0.67254601]
CV macro recall: [0.36747261 0.39524594 0.380243   0.39372381 0.39694151]
CV weighted recall: [0.65057471 0.6651341  0.66871166 0.66871166 0.67254601]

Mean accuracy: 0.6651356274827822
Mean macro recall: 0.3867253754130049
Mean weighted recall: 0.6651356274827822
